In [ ]:
!pip install -q --upgrade Pillow timm albumentations kaggle scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 92.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import json, os

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "suyashjaiswal005", "key": "KGAT_c2842e6d8262e2a37714114438b4f390"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Dataset 1 - StyleGAN fakes (large, works well)
print("Downloading Dataset 1: 140k GAN faces...")
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data/gan
!unzip -q /content/data/gan/140k-real-and-fake-faces.zip -d /content/data/gan

# Dataset 2 - DFDC face swaps (large, diverse)
print("Downloading Dataset 2: DFDC face swaps...")
!kaggle datasets download -d dagnelies/deepfake-faces -p /content/data/dfdc
!unzip -q /content/data/dfdc/deepfake-faces.zip -d /content/data/dfdc

# Dataset 3 - Morphed faces
print("Downloading Dataset 3: Morphed faces...")
!kaggle datasets download -d hamzaboulahia/hardfakevsrealfaces -p /content/data/morph
!unzip -q /content/data/morph/hardfakevsrealfaces.zip -d /content/data/morph

print("\n✅ All downloads done!")

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:31<00:00, 126MB/s]

Dataset URL: https://www.kaggle.com/datasets/dagnelies/deepfake-faces
License(s): other
100% 433M/433M [00:06<00:00, 71.7MB/s]

Dataset URL: https://www.kaggle.com/datasets/hamzaboulahia/hardfakevsrealfaces
License(s): CC0-1.0
100% 15.3M/15.3M [00:00<00:00, 46.3MB/s]


✅ All downloads done!


In [ ]:
import os

def print_structure(base, name, max_depth=3):
    print(f"\n=== {name} ===")
    for root, dirs, files in os.walk(base):
        level = root.replace(base, "").count(os.sep)
        if level >= max_depth:
            continue
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/ ({len(files)} files)")

print_structure("/content/data/gan",   "GAN Dataset")
print_structure("/content/data/dfdc",  "DFDC Dataset")
print_structure("/content/data/morph", "Morph Dataset")


=== GAN Dataset ===
gan/ (4 files)
  real_vs_fake/ (0 files)
    real-vs-fake/ (0 files)

=== DFDC Dataset ===
dfdc/ (2 files)
  faces_224/ (95634 files)

=== Morph Dataset ===
morph/ (2 files)
  real/ (589 files)
  fake/ (700 files)


In [ ]:
import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
EPOCHS     = 5
IMG_SIZE   = 224

print(f"Device: {DEVICE}")
print(f"Torch: {torch.__version__}")

Device: cuda
Torch: 2.10.0+cu128


In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.GaussNoise(p=0.2),
    A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),
    A.Normalize(mean=[0.5]*3, std=[0.5]*3),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5]*3, std=[0.5]*3),
    ToTensorV2()
])

/tmp/ipykernel_3473/897208574.py:6: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),


In [ ]:
all_samples = []

# ── Dataset 1: GAN (140k) ──────────────────────────────────────
print("\n[1] GAN Dataset")
all_samples += collect_from_folder("/content/data/gan/real_vs_fake/real-vs-fake/train/real", label=1, limit=20000)
all_samples += collect_from_folder("/content/data/gan/real_vs_fake/real-vs-fake/train/fake", label=0, limit=20000)

# ── Dataset 2: DFDC ───────────────────────────────────────────
# faces_224 has 95k files - check if subfolders exist
print("\n[2] DFDC Dataset")
dfdc_base = "/content/data/dfdc/faces_224"
dfdc_contents = os.listdir(dfdc_base)
print(f"  DFDC contents sample: {dfdc_contents[:10]}")

# Check if it has subfolders or flat files
if "real" in dfdc_contents or "fake" in dfdc_contents:
    all_samples += collect_from_folder(f"{dfdc_base}/real", label=1, limit=15000)
    all_samples += collect_from_folder(f"{dfdc_base}/fake", label=0, limit=15000)
else:
    # Flat folder - filenames usually contain 'real'/'fake' label
    real_files = [f for f in dfdc_contents if "real" in f.lower()]
    fake_files = [f for f in dfdc_contents if "fake" in f.lower()]
    print(f"  Flat folder — real files: {len(real_files)} | fake files: {len(fake_files)}")
    all_samples += [(os.path.join(dfdc_base, f), 1) for f in real_files[:15000]]
    all_samples += [(os.path.join(dfdc_base, f), 0) for f in fake_files[:15000]]

# ── Dataset 3: Morphed faces ──────────────────────────────────
print("\n[3] Morph Dataset")
all_samples += collect_from_folder("/content/data/morph/real", label=1)
all_samples += collect_from_folder("/content/data/morph/fake", label=0)

print(f"\n✅ Total samples: {len(all_samples)}")
real_count = sum(1 for _, l in all_samples if l == 1)
fake_count = sum(1 for _, l in all_samples if l == 0)
print(f"   Real: {real_count} | Fake: {fake_count}")


[1] GAN Dataset
  REAL from /content/data/gan/real_vs_fake/real-vs-fake/train/real: 20000
  FAKE from /content/data/gan/real_vs_fake/real-vs-fake/train/fake: 20000

[2] DFDC Dataset
  DFDC contents sample: ['eefziuudfy.jpg', 'btpaxvxwid.jpg', 'uqnsynghnc.jpg', 'ktnovgcglh.jpg', 'bcqptrgmdt.jpg', 'hdkwxvnwbp.jpg', 'glghscbqzn.jpg', 'zbrmezlilf.jpg', 'kktzicobyz.jpg', 'nofbozkher.jpg']
  Flat folder — real files: 2 | fake files: 2

[3] Morph Dataset
  REAL from /content/data/morph/real: 589
  FAKE from /content/data/morph/fake: 700

✅ Total samples: 41293
   Real: 20591 | Fake: 20702


In [ ]:
import os

# Check for any metadata files
dfdc_root = "/content/data/dfdc"
for root, dirs, files in os.walk(dfdc_root):
    for f in files:
        if f.endswith((".csv", ".json", ".txt")):
            print(os.path.join(root, f))

/content/data/dfdc/metadata.csv


In [ ]:
import pandas as pd

df = pd.read_csv("/content/data/dfdc/metadata.csv")
print(df.shape)
print(df.head(10))
print(df.columns.tolist())
print(df["label"].value_counts() if "label" in df.columns else "no label column")

(95634, 5)
        videoname  original_width  original_height label        original
0  aznyksihgl.mp4             129              129  FAKE  xnojggkrxt.mp4
1  gkwmalrvcj.mp4             129              129  FAKE  hqqmtxvbjj.mp4
2  lxnqzocgaq.mp4             223              217  FAKE  xjzkfqddyk.mp4
3  itsbtrrelv.mp4             186              186  FAKE  kqvepwqxfe.mp4
4  ddvgrczjno.mp4             155              155  FAKE  pluadmqqta.mp4
5  gaffodreks.mp4             223              223  FAKE  vwtztxpljw.mp4
6  adovxmilsk.mp4              90               89  FAKE  cqkarxqzxx.mp4
7  lldtrmyzbe.mp4             108              107  FAKE  yeqqlbhgze.mp4
8  sjgbiywngj.mp4             185              186  FAKE  dbjmhpklbv.mp4
9  rumyjhyayo.mp4              89               90  FAKE  yhwkxfpoio.mp4
['videoname', 'original_width', 'original_height', 'label', 'original']
label
FAKE    79341
REAL    16293
Name: count, dtype: int64


In [ ]:
import pandas as pd
import os

df = pd.read_csv("/content/data/dfdc/metadata.csv")
dfdc_base = "/content/data/dfdc/faces_224"

# Check if stripping .mp4 matches jpg filenames
sample_video = df["videoname"].iloc[0]  # e.g. aznyksihgl.mp4
stem = sample_video.replace(".mp4", "")  # aznyksihgl
jpg_path = os.path.join(dfdc_base, stem + ".jpg")
print(f"Video: {sample_video}")
print(f"Looking for: {jpg_path}")
print(f"Exists: {os.path.exists(jpg_path)}")

# Build label map
df["stem"] = df["videoname"].str.replace(".mp4", "", regex=False)
label_map = dict(zip(df["stem"], df["label"]))  # stem → REAL/FAKE

# Test a few actual jpg files
sample_jpgs = os.listdir(dfdc_base)[:5]
for jpg in sample_jpgs:
    stem = jpg.replace(".jpg", "")
    print(f"{jpg} → {label_map.get(stem, 'NOT FOUND')}")

Video: aznyksihgl.mp4
Looking for: /content/data/dfdc/faces_224/aznyksihgl.jpg
Exists: True
eefziuudfy.jpg → FAKE
btpaxvxwid.jpg → REAL
uqnsynghnc.jpg → FAKE
ktnovgcglh.jpg → REAL
bcqptrgmdt.jpg → FAKE


In [ ]:
import pandas as pd

# Build DFDC label map
df = pd.read_csv("/content/data/dfdc/metadata.csv")
df["stem"] = df["videoname"].str.replace(".mp4", "", regex=False)
label_map = dict(zip(df["stem"], df["label"]))

dfdc_base = "/content/data/dfdc/faces_224"
dfdc_real, dfdc_fake = [], []

for jpg in os.listdir(dfdc_base):
    if not jpg.lower().endswith(".jpg"):
        continue
    stem = jpg.replace(".jpg", "")
    label = label_map.get(stem)
    if label == "REAL":
        dfdc_real.append((os.path.join(dfdc_base, jpg), 1))
    elif label == "FAKE":
        dfdc_fake.append((os.path.join(dfdc_base, jpg), 0))

print(f"DFDC Real: {len(dfdc_real)} | Fake: {len(dfdc_fake)}")

all_samples = []

# ── Dataset 1: GAN ────────────────────────────────────────────
print("\n[1] GAN Dataset")
all_samples += collect_from_folder("/content/data/gan/real_vs_fake/real-vs-fake/train/real", label=1, limit=20000)
all_samples += collect_from_folder("/content/data/gan/real_vs_fake/real-vs-fake/train/fake", label=0, limit=20000)

# ── Dataset 2: DFDC ───────────────────────────────────────────
print("\n[2] DFDC Dataset")
all_samples += dfdc_real[:15000]
all_samples += dfdc_fake[:15000]

# ── Dataset 3: Morph ──────────────────────────────────────────
print("\n[3] Morph Dataset")
all_samples += collect_from_folder("/content/data/morph/real", label=1)
all_samples += collect_from_folder("/content/data/morph/fake", label=0)

print(f"\n✅ Total samples: {len(all_samples)}")
real_count = sum(1 for _, l in all_samples if l == 1)
fake_count = sum(1 for _, l in all_samples if l == 0)
print(f"   Real: {real_count} | Fake: {fake_count}")

DFDC Real: 16293 | Fake: 79341

[1] GAN Dataset
  REAL from /content/data/gan/real_vs_fake/real-vs-fake/train/real: 20000
  FAKE from /content/data/gan/real_vs_fake/real-vs-fake/train/fake: 20000

[2] DFDC Dataset

[3] Morph Dataset
  REAL from /content/data/morph/real: 589
  FAKE from /content/data/morph/fake: 700

✅ Total samples: 71289
   Real: 35589 | Fake: 35700


In [ ]:
train_samples, val_samples = train_test_split(
    all_samples, test_size=0.1, random_state=42, shuffle=True
)

train_dataset = CombinedDataset(train_samples, transform=train_transform)
val_dataset   = CombinedDataset(val_samples,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 64160 | Val: 7129


In [ ]:
model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=2)
model = model.to(DEVICE)
print("Model ready ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Model ready ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")

    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
    print(f"Train Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/len(train_dataset)*100:.2f}%")

    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Validating"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
    val_acc = correct / len(val_dataset)
    print(f"Val Acc: {val_acc*100:.2f}%")

    scheduler.step()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/drive/MyDrive/deepfake_model_combined.pth")
        print("✅ Best model saved to Drive!")

print(f"\n🏁 Done! Best Val Acc: {best_val_acc*100:.2f}%")

Mounted at /content/drive

--- Epoch 1/5 ---


Training: 100%|██████████| 2005/2005 [15:44<00:00,  2.12it/s]


Train Loss: 0.5427 | Acc: 71.92%


Validating: 100%|██████████| 223/223 [00:27<00:00,  8.15it/s]


Val Acc: 85.30%
✅ Best model saved to Drive!

--- Epoch 2/5 ---


...

Val Acc: 93.31%
✅ Best model saved to Drive!

--- Epoch 5/5 ---


Training: 100%|██████████| 2005/2005 [16:01<00:00,  2.09it/s]


Train Loss: 0.2125 | Acc: 89.30%


Validating: 100%|██████████| 223/223 [00:26<00:00,  8.27it/s]

Val Acc: 93.31%

🏁 Done! Best Val Acc: 93.31%


In [4]:
import torch
import timm
import numpy as np
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "/content/drive/MyDrive/deepfake_model.pth"

# Load model
model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("Model loaded ✅")

# Load test set
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_dataset = datasets.ImageFolder("/content/data/real-vs-fake/test", transform=transform_test)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {test_dataset.class_to_idx}")

Mounted at /content/drive
Model loaded ✅


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/real-vs-fake/test'